In [34]:
#!/usr/bin/env python3
import whisper
from whisper.audio import SAMPLE_RATE
from pathlib import Path

# 1. Load model (choose 'tiny', 'base', 'small', 'medium', 'large')
size = "small.en"
model = whisper.load_model(size)

audio_path = Path("long_cumberbatch.wav")
srt_path = f"{size}_{audio_path.with_suffix('.srt')}"

# 2. Load the full waveform (mono @ SAMPLE_RATE), no trimming here
audio = whisper.load_audio(str(audio_path))  # → np.ndarray, length ∝ duration

# 3. Transcribe in fixed-length chunks
SEGMENT = 30  # seconds
step    = SAMPLE_RATE * SEGMENT
subs    = []  # list of (start, end, text)

for i in range(0, len(audio), step):
    chunk = audio[i : min(i + step, len(audio))]  # no padding
    result = model.transcribe(chunk, fp16=False, verbose=False)
    for seg in result["segments"]:
        start = seg["start"] + (i / SAMPLE_RATE)
        end   = seg["end"]   + (i / SAMPLE_RATE)
        text  = seg["text"].strip()
        subs.append((start, end, text))

# 4. Write out SRT
def fmt_ts(seconds):
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

with open(srt_path, "w") as f:
    for idx, (start, end, text) in enumerate(subs, 1):
        f.write(f"{idx}\n")
        f.write(f"{fmt_ts(start)} --> {fmt_ts(end)}\n")
        f.write(text + "\n\n")

print(f"Wrote subtitles to {srt_path}")

100%|██████████| 2366/2366 [00:00<00:00, 2787.79frames/s]

Wrote subtitles to small.en_long_cumberbatch.srt
